### Consolidating Data

In [ ]:
import os
import sys

notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date
from src.ingestion import bcb, tesouro
from src.curves import bootstrap, models, utils

# Avoid Restarting Kernel
%load_ext autoreload
%autoreload 2

# Load data

data_bcb = bcb.fetch_all(start_date=date(2024, 1, 1))
data_tesouro = tesouro.fetch_all(start_date=date(2024, 1, 1))

# Bootstrap the Real Curve (NTN-B)
df_ntnb = bootstrap.bootstrap_ntnb_principal(
    data_tesouro['ntnb_zero'], 
    data_bcb['vna_ntn_b']
)

# Select date
target_date = df_ntnb['date'].max()
df_snapshot = df_ntnb[df_ntnb['date'] == target_date].sort_values('du')

print(f"Visualizing Curve for: {target_date.date()}")

2026-06-09 08:13:31,250 [INFO] Fetching series 12. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 08:13:31,522 [INFO] Series 12: 610 records fetched.
2026-06-09 08:13:31,525 [INFO] Fetching series 433. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 08:13:31,678 [INFO] Series 433: 28 records fetched.
2026-06-09 08:13:31,680 [INFO] Fetching series 226. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 08:13:31,908 [INFO] Series 226: 904 records fetched.
2026-06-09 08:13:31,910 [INFO] Fetching series 432. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 08:13:32,114 [INFO] Series 432: 891 records fetched.
2026-06-09 08:13:32,118 [INFO] Fetching series 11. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 08:13:32,318 [INFO] Series 11: 611 records fetched.
2026-06-09 08:13:32,322 [INFO] Fetching series 12466. From 2024-01-01 to 2026-06-09 (attempt 1)
2026-06-09 08:13:33,578 [WARNING] HTTP error on attempt 1: 404 Client Error: Not Found for url: https://api.bcb.gov.br/

ReadTimeout: HTTPSConnectionPool(host='api.bcb.gov.br', port=443): Read timed out. (read timeout=30)

### Fitting Nelson-Siegel Model
Convert business days to year fraction and fit Model

In [ ]:
# Time (t) in years and Yields (y)
t_obs = df_snapshot['du'].values / 252
y_obs = df_snapshot['yield_calculated'].values

# Fit parameters
b0, b1, b2, tau = models.fit_nelson_siegel(t_obs, y_obs)

print(f"NS Parameters:\n Level (b0): {b0:.3f}\n Slope (b1): {b1:.3f}\n Curvature (b2): {b2:.3f}\n Tau: {tau:.3f}")

### Visualizing the Curve
How the parametric model smooths the market curve. 

In [ ]:
# Generate continuous curve for plotting (0 to 10 years)
t_curve = np.linspace(0.1, 10, 100)
y_curve = models.nelson_siegel(t_curve, b0, b1, b2, tau)

fig = go.Figure()

# Market Points
fig.add_trace(go.Scatter(
    x=t_obs, y=y_obs, 
    mode='markers', 
    name='Market Data (NTN-B)',
    marker=dict(size=10, color='red')
))

# Nelson-Siegel Fit
fig.add_trace(go.Scatter(
    x=t_curve, y=y_curve, 
    mode='lines', 
    name='Nelson-Siegel Fit',
    line=dict(dash='dash', color='blue')
))

fig.update_layout(
    title=f"IPCA Real Yield Curve - {target_date.date()}",
    xaxis_title="Maturity (Years)",
    yaxis_title="Annual Real Yield (%)",
    template="plotly_white"
)

fig.show()